<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

<span style="float:right">
<strong>Credits:</strong>
Ce tutoriel est une adaptation du <a href="https://wiki.ros.org/ROS/Tutorials/">tutoriel officiel de ROS</a>, initialement publié sous la licence <a href="http://creativecommons.org/licenses/by/3.0/">Creative Commons Attribution 3.0</a>.
<span>

# Comprendre les ROS Nodes

 Ce tutoriel présente l'utilisation de [roscore](https://wiki.ros.org/roscore), [rosnode](https://wiki.ros.org/rosnode), et [rosrun](https://wiki.ros.org/rosrun), outils CLI.

## Suivre le tutoriel

Pour cette partie, nous utiliserons un environnement graphique **connecté à votre robot**. Autrement dit, il utilise le **ROS_MASTER** de votre robot.
Pour le lancer, vous pouvez exécuter la commande suivante dans le terminal **de votre ordinateur portable** (et non dans le terminal de VS Code !) :

    dts code vnc -R [!ROBOT_NAME]

Cela lancera un conteneur avec ROS installé.
Un lien apparaîtra dans le terminal et, en cliquant dessus, vous ouvrirez un environnement VNC desktop :
```

================================================================
|                                                              |
|    VNC running at http://127.0.0.1:32780                     |
|                                                              |
================================================================
```

(Le lien peut être différent.)

Sur le bureau, vous trouverez plusieurs icônes. Pour ouvrir un terminal, vous pouvez utiliser `LXTerminal`.

![desktop_icons](../assets/desktop_icons.png)

## Nodes

Un node n'est en réalité rien de plus qu'un fichier exécutable au sein d'un package ROS. Les ROS nodes utilisent une librairie cliente ROS pour communiquer avec d'autres nodes. Les nodes peuvent publish des messages sur un topic ou subscribe a un topic. Ils peuvent également fournir ou utiliser un service (Cela ne sera pas abordé dans ce tutoriel).

Les librairies clientes ROS permettent aux nodes écrits dans différents langages de programmation de communiquer :

- `rospy` : librairie cliente Python
- `roscpp` : librairie cliente C++

## roscore

`roscore` est le premier élément à exécuter lors de l'utilisation de ROS. Dans notre cas, il est déjà en cours d'exécution sur le robot, nous n'avons donc pas besoin de le démarrer.

## Utiliser `rosnode`

Ouvrez un **nouveau terminal** et utilisons la commande `rosnode` pour voir quels nœuds sont actuellement en cours d'exécution.

La commande `rosnode` affiche des informations sur les ROS nodes actuellement en cours d'exécution. La commande `rosnode list` répertorie ces nœuds actifs :

    rosnode list

Vous verrez une liste des nodes qui roules déjà sur votre robot, notamment `/rosout`, qui est toujours actif car il collecte et enregistre les messages de débogage des nodes, ainsi que les nodes des interfaces des capteurs et des actuateurs, que nous utilisons pour lire les données des capteurs et envoyer des commandes d'actionnement afin que le robot puisse agir. Pour l'instant, nous n'avons pas besoin de nous préoccuper des détails de la plupart d'entre eux.

La commande `rosnode info` renvoie des informations sur un node spécifique.

    rosnode info /ROBOTNAME/camera_node

Où `ROBOTNAME` doit être remplacé par le nom de votre robot (le nom que vous avez indiqué après l'option `-R` lors de l'exécution de la commande `dts code vnc` ci-dessus).
Cela nous a fourni des informations supplémentaires sur le node de la caméra.

```bash
--------------------------------------------------------------------------------
Node [/ROBOTNAME/camera_node]
Publications: 
 * /rosout [rosgraph_msgs/Log]
 * /ROBOTNAME/camera_node/camera_info [sensor_msgs/CameraInfo]
 * /ROBOTNAME/camera_node/image/compressed [sensor_msgs/CompressedImage]

Subscriptions: None

Services: 
 * /vlx/camera_node/get_loggers
 * /vlx/camera_node/get_parameters_list
 * /vlx/camera_node/request_parameters_update
 * /vlx/camera_node/set_logger_level
 * /vlx/camera_node/switch


contacting node http://ROBOTNAME.local:33271/ ...
Pid: 5538
Connections: ...
```

**Remarque :** Ne vous inquiétez pas si le résultat obtenu n'est pas exactement identique à celui présenté ci-dessus, mais il devrait être globalement similaire.

Cela nous renseigne sur certaines caractéristiques de ce node, notamment sur le fait qu'il publie des données sur trois topics différents :

 - `/rosout` qui est de type [rosgraph_msgs/Log](https://docs.ros.org/en/noetic/api/rosgraph_msgs/html/msg/Log.html)
 - `/ROBOTNAME/camera_node/camera_info` qui est de type [sensor_msgs/CameraInfo](https://docs.ros.org/en/noetic/api/sensor_msgs/html/msg/CameraInfo.html)
 - `/ROBOTNAME/camera_node/image/compressed` qui est de type [/ROBOTNAME/camera_node/image/compressed](https://docs.ros.org/en/melodic/api/sensor_msgs/html/msg/CompressedImage.html)

Ce node ne subscribe à aucun topics (ce qui est assez typique pour un pilote de capteur). De plus, ce node propose quelques [services](https://wiki.ros.org/Services). Les détails de ces services ne sont pas essentiels pour l'instant, mais vous pouvez les considérer comme des fonctions spécifiques mises à disposition par ce node et pouvant être appelées depuis n'importe où (ils sont les RPCs).

## Utiliser `rosrun`

La commande `rosrun` vous permet d'utiliser le nom du package pour exécuter directement un _node_ au sein d'un _package_ (sans avoir à connaître le chemin d'accès du package).

Usage:

    rosrun [package_name] [node_name]

Nous pouvons donc maintenant exécuter le node `turtlesim_node` du package `turtlesim`.

Dans un nouveau (dans l'environment `VNC`):

    rosrun turtlesim turtlesim_node

Vous verrez la fenêtre de turtlesim :

![turtlesim](../assets/turtlesim.png) 

**REMARQUE :** La tortue peut avoir une apparence différente dans votre fenêtre turtlesim. Ne vous inquiétez pas, il existe de nombreux types de tortues et la vôtre est une surprise !

Dans un nouveau terminal, exécutez la commande suivante :
    rosnode list

Vous constaterez que `turtlesim` est désormais inclus dans la liste.

Fermez la fenêtre turtlesim pour arrêter le node (ou revenez au terminal où vous avez exécuté la commande `rosrun turtlesim turtlesim_node` et utilisez `Ctrl+C`). Relançons maintenant le programme, mais cette fois-ci, utilisons un argument de remappage pour modifier le nom du node :


    rosrun turtlesim turtlesim_node __name:=my_turtle

Maintenant, si nous revenons en arrière et utilisons la commande `rosnode list` :

    rosnode list

Nous voyons notre nouveau nœud `/my_turtle`. Utilisons une autre commande `rosnode`, `ping`, pour vérifier qu'il est bien actif :

    rosnode ping my_turtle

```bash
    rosnode: node is [/my_turtle]
    pinging /my_turtle with a timeout of 3.0s
    xmlrpc reply from http://aqy:42235/     time=1.152992ms
    xmlrpc reply from http://aqy:42235/     time=1.120090ms
    xmlrpc reply from http://aqy:42235/     time=1.700878ms
    xmlrpc reply from http://aqy:42235/     time=1.127958ms
```

## Sommaire

Ce qui a été abordé :

- `roscore` = ros+core : le `master` (fournit le service de noms pour ROS) + `rosout` (`stdout`/ `stderr`) + parameter server (le parameter server sera présenté ultérieurement)
- `rosnode` = ros+node : outil ROS permettant d'obtenir des informations sur un node.
- `rosrun` = ros+run : exécute un node à partir d'un package donné.

Maintenant que vous comprenez le fonctionnement des nodes ROS, voyons comment fonctionnent les topics ROS. N'hésitez pas à appuyer sur `Ctrl` + `C` pour arrêter `turtlesim_node`.

Passons au [notebook suivant sur les ROS topics](../notebooks/05_ros_topics.ipynb)